In [1]:
import os
import numpy as np
import rasterio
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape

In [ ]:
# Open the NetCDF file
base_path = '/work/hawkinslab/jfhawkin/SH-BART/data/vulcan/Vulcan_V3_Annual_Emissions_1741/'
file_names = ['Vulcan_v3_US_annual_1km_residential_mn','Vulcan_v3_US_annual_1km_onroad_mn',
               'Vulcan_v3_US_annual_1km_commercial_mn','Vulcan_v3_US_annual_1km_industrial_mn',
               'Vulcan_v3_US_annual_1km_total_mn']

fill_value = -9999

for f in file_names:
    nc_file = os.path.join(base_path,'data',f+'.nc4')
    with rasterio.open(nc_file) as src:
        raster_data = src.read(6) # Take the fifth layer = 2019 data
        raster_data = raster_data.astype('float32') # shapes cannot handle 'float64'

        # Handle missing values by creating a mask
        mask = raster_data!=fill_value

        # Define the transformation
        transform = src.transform

        # Convert raster data to shapes
        shapes_gen = shapes(raster_data, mask=mask, transform=transform)

        # Create a list of records with geometry and value
        records = []
        for geom, value in shapes_gen:
            if np.isfinite(value):  # Filter out NaN or invalid values
                records.append({
                    'geometry': shape(geom),
                    'value': float(value)
                })
        gdf = gpd.GeoDataFrame(records, crs=src.crs)  # You can change the CRS to match your data
        shp_file = os.path.join(base_path,'shp',f+'_2019.shp')
        print(shp_file)
        gdf.to_file(shp_file)

/work/hawkinslab/jfhawkin/SH-BART/data/vulcan/Vulcan_V3_Annual_Emissions_1741/shp/Vulcan_v3_US_annual_1km_residential_mn_2015.shp
/work/hawkinslab/jfhawkin/SH-BART/data/vulcan/Vulcan_V3_Annual_Emissions_1741/shp/Vulcan_v3_US_annual_1km_onroad_mn_2015.shp
/work/hawkinslab/jfhawkin/SH-BART/data/vulcan/Vulcan_V3_Annual_Emissions_1741/shp/Vulcan_v3_US_annual_1km_commercial_mn_2015.shp
/work/hawkinslab/jfhawkin/SH-BART/data/vulcan/Vulcan_V3_Annual_Emissions_1741/shp/Vulcan_v3_US_annual_1km_industrial_mn_2015.shp
